# EXP-2026-008 / Q5-E - Leg 2 failure mechanism audit

Implementation only. This notebook is committed **unexecuted**: every output
cell is empty and no cell has an execution count.

Running the audit on the registered artifacts needs a **separate user
approval that does not exist yet**. Two independent switches keep it closed:
`OPEN_REGISTERED_DATA` defaults to `False`, and the production route also
requires an explicit approval token. Either one alone refuses.

Read the frozen design first:
`experiments/specs/EXP-2026-008-q5e-leg2-failure-mechanism-audit.md`.

Order of the cells below is the order of the contract: setup, staleness
guard, constants, switches, per-stage announcement, production route.

In [ ]:
# 1. Setup. Declared dependencies are checked before anything is read.
REPO = '/content/repo'
import os, sys
if not os.path.exists(REPO):
    REPO = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(REPO, 'mit-bih'))

import q5d_order_preserving_beat_join as BJ
import q5e_leg2_failure_mechanism_audit as Q5E

print('Q5-E module :', Q5E.__file__)
print('frozen Q5-D :', BJ.__file__)

In [ ]:
# 2. Staleness guard. A version integer is defeated by forgetting to bump it,
#    so assert the capabilities actually used - on BOTH modules.
MISSING_Q5E = [n for n in Q5E.module_capabilities() if not hasattr(Q5E, n)]
NEED_BJ = ('candidate_edges', 'match_record', 'rule_fingerprint',
           'cache_expected_files', 'hash_file_set', 'RecordSequence')
MISSING_BJ = [n for n in NEED_BJ if not hasattr(BJ, n)]
assert not MISSING_Q5E, f'stale Q5-E clone, missing {MISSING_Q5E}'
assert not MISSING_BJ, f'stale Q5-D clone, missing {MISSING_BJ}'
assert BJ.rule_fingerprint() == Q5E.REGISTERED_RULE_FINGERPRINT, \
    'frozen Q5-D rule fingerprint moved'
print('capabilities OK; frozen fingerprint', BJ.rule_fingerprint())

In [ ]:
# 3. Constants card. This opens nothing and is not a result.
print(Q5E.design_card())
print()
print(Q5E.assert_implementation_only())

In [ ]:
# 4. Switches. Both default to closed and stay that way until the separate
#    execution approval exists. Do not edit these to "try it out".
MODE = Q5E.MODE_DESIGN
APPROVAL = None
OPEN_REGISTERED_DATA = False

print('MODE                :', MODE)
print('approval present    :', Q5E.execution_is_approved(APPROVAL))
print('OPEN_REGISTERED_DATA:', OPEN_REGISTERED_DATA)
print()
print(Q5E.APPROVAL_NOTE)

In [ ]:
# 5. Runtime dependency table for the selected stage, before any work.
REPORT = Q5E.check_runtime_dependencies(MODE)
for name, (why, pinned) in sorted(Q5E.RUNTIME_DEPENDENCIES.items()):
    print(f'{name:12s} {pinned or "-":8s} {why}')
print()
print('required for this stage:', REPORT['required'])
print('missing                :', REPORT['missing'])
if REPORT['missing']:
    print('install first          :', REPORT['pip_install'])

In [ ]:
# 6. Every stage announces RUN or SKIP with its reason. A stage that quietly
#    does nothing must never look like a stage that passed.
STAGES = ('QA', 'M0', 'M1', 'M2', 'M3', 'M4-gate', 'controls', 'decision')
WILL_RUN = {s: Q5E.stage_should_run(s, MODE, APPROVAL) for s in STAGES}
print()
print('stages that would run:', [s for s, v in WILL_RUN.items() if v])

In [ ]:
# 7. Figure contract. Titles and axes are ASCII so Colab cannot render a
#    missing glyph into a plot that is then cited as a result.
SPECS = Q5E.figure_specs(m4_ok=False)
Q5E.assert_ascii_labels(SPECS)
for spec in SPECS:
    print(f"{spec['file']:44s} {spec['title']}")
print()
print('figure 7 is produced only when the M4 gate passes; its absence is\n'
      'recorded in the bundle together with the reason.')

In [ ]:
# 8. Production route. This is the ONLY path to a Q5-E decision.
#    With the switches above it refuses, which is the intended state of this
#    notebook as committed.
BUNDLE_DIR = ''
MAMBA_PATH = ''
CACHE_DIR = ''
MITDB_DIR = ''
OUT_DIR = ''

if Q5E.stage_should_run('run_audit', MODE, APPROVAL):
    RESULT = Q5E.run_audit(BUNDLE_DIR, MAMBA_PATH, CACHE_DIR, MITDB_DIR,
                           OUT_DIR, approval=APPROVAL,
                           open_registered_data=OPEN_REGISTERED_DATA)
    print('decision:', RESULT['decision'])
else:
    print('run_audit not started. Nothing above this line is a measurement.')

## What this notebook may not do

No registered aggregation, no detector reproduction, no beat-join re-run, no
per-beat label of the held-out split, no model score, no association, no
training, and no write to an existing bundle. The audit reports **associated
mechanisms** only; it never states that an association is a cause, and it
licenses no change to the frozen Q5-D join rule.

Next steps, in order: this implementation PR is reviewed, the user separately
approves execution, and only then may the stages above run and write one new
timestamped bundle.